In [1]:

import json
import pennylane as qml
from pennylane import qchem
import pennylane.numpy as np

symbols = ["H", "H", "H"]

def h3_ground_energy(bond_length):
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.
    
    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
        state of the H3+ molecule with the given bond length.
    """
    coordinates = np.array([
        [0, 0, 0],
        [bond_length, 0, 0],
        [bond_length/2, bond_length*np.sqrt(3)/2, 0]
    ], requires_grad=False)
    H, qubits = qchem.molecular_hamiltonian(symbols, coordinates, charge=1)
    electrons = 2 
    hf_state = qchem.hf_state(electrons, qubits)
    dev = qml.device('default.qubit', wires=qubits)
    @qml.qnode(dev)
    def circuit(params, wires):
        qml.BasisState(hf_state, wires=wires)
        qml.DoubleExcitation(params[0], wires=[0, 1, 2, 3])
        qml.DoubleExcitation(params[1], wires=[0, 1, 4, 5])
        return qml.expval(H)
    params = np.array([0.0, 0.0], requires_grad=True)
    optimizer = qml.GradientDescentOptimizer(stepsize=0.4)
    max_iterations = 200
    conv_tol = 1e-06
    energy = [circuit(params, range(qubits))]
    for n in range(max_iterations):
        params, prev_energy = optimizer.step_and_cost(circuit, params, wires=range(qubits))
        energy.append(circuit(params, wires=range(qubits)))
        conv = np.abs(energy[-1] - prev_energy)
        if conv <= conv_tol:
            break
    return energy[-1].item()

def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol = 1e-4), "Not the correct ground energy"

test_cases = [
    ('1.5', '-1.232574'),
    ('0.8', '-0.3770325')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)

    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

        else:
            print("Correct!")


Running test case 0 with input '1.5'...


Correct!
Running test case 1 with input '0.8'...


Correct!
